<a href="https://colab.research.google.com/github/Renata-Nascimento/Projeto-U02-CD/blob/main/notebooks/02_limpeza_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, re, sys, glob, unicodedata
import numpy as np
import pandas as pd
import requests

base_ibge = pd.read_csv('https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/base_municipios_2020_2023.csv')
base_sefaz = pd.read_csv('https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/repasses_2020_2023.csv')

In [2]:
# Extraindo as colunas únicas diretamente do dataframe
municipios = base_ibge[['cod_ibge', 'municipio', 'mesorregiao', 'regiao_intermediaria']].drop_duplicates().reset_index(drop=True)

print(len(municipios), 'municípios')
municipios.head()

167 municípios


,cod_ibge,municipio,mesorregiao,regiao_intermediaria
0,2400109,Acari,Central Potiguar,Caicó
1,2400208,Assú,Oeste Potiguar,Mossoró
2,2400307,Afonso Bezerra,Central Potiguar,Mossoró
3,2400406,Água Nova,Oeste Potiguar,Mossoró
4,2400505,Alexandria,Oeste Potiguar,Mossoró


In [3]:
def chave(nome):
    """'Olho d'Água do Borges' -> 'OLHODAGUADOBORGES'"""
    s = unicodedata.normalize('NFKD', str(nome)).encode('ascii', 'ignore').decode().upper()
    return re.sub(r'[^A-Z]', '', s)

DE_PARA = {
    'ARES': 'AREZ',
    'BOASAUDE': 'JANUARIOCICCO',                 # nome popular (oficial: Januário Cicco)
    'CAICARADORIODOSVENTOS': 'CAICARADORIODOVENTO',
    'GOVDIXSEPTROSADO': 'GOVERNADORDIXSEPTROSADO',
    'IPANGUASSU': 'IPANGUACU',
    'OLHODAGUADOSBORGES': 'OLHODAGUADOBORGES',
    'RUIBARBOSA': 'RUYBARBOSA',
    'SAOMIGUELDETOUROS': 'SAOMIGUELDOGOSTOSO',   # nome antigo
    'SENELOIDESOUZA': 'SENADORELOIDESOUZA',
    'SENGEORGINOAVELINO': 'SENADORGEORGINOAVELINO',
}

COD_POR_CHAVE = dict(zip(municipios['municipio'].map(chave), municipios['cod_ibge']))

In [4]:
base = base_ibge.merge(base_sefaz, on='cod_ibge')
base.head()

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano_x,rep_icms,rep_ipi,rep_ipva,rep_royalties,rep_total,...,cad_pessoas,rep_total_real,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,ano_y,tipo,valor
0,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2020,ICMS,3.116225e+06
1,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2020,IPVA,3.850488e+05
2,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2020,IPI,3.279713e+03
3,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2020,ROYALTIES,4.459055e+04
4,2400109,Acari,Central Potiguar,Caicó,2020,3.116225e+06,3279.71321,385048.78,44590.552676,3.549144e+06,...,6675,4.393441e+06,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,2021,ICMS,3.576256e+06


### Limpeza e Unificação das Bases de Dados (IBGE & SEFAZ)

Vamos selecionar apenas as colunas de interesse de cada tabela:
* **IBGE**: `cod_ibge`, `municipio`, `mesorregiao`, `regiao_intermediaria`, `ano`, `pib_real_mil`, `rep_per_capita`, `pib_per_capita`, `pct_baixa_renda`, `porte`.
* **SEFAZ**: `cod_ibge`, `ano`, `tipo`, `valor`.

In [5]:
import pandas as pd

# 1. Carregar os dados originais do GitHub
raw_ibge = pd.read_csv('https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/base_municipios_2020_2023.csv')
raw_sefaz = pd.read_csv('https://raw.githubusercontent.com/Renata-Nascimento/Projeto-U02-CD/main/saida/repasses_2020_2023.csv')

# 2. Selecionar apenas as colunas de interesse para a nossa análise
colunas_interesse_ibge = [
    'cod_ibge', 'municipio', 'mesorregiao', 'regiao_intermediaria',
    'ano', 'pib_real_mil', 'rep_per_capita', 'pib_per_capita',
    'pct_baixa_renda', 'porte'
]
colunas_interesse_sefaz = ['cod_ibge', 'ano', 'tipo', 'valor']

df_ibge_clean = raw_ibge[colunas_interesse_ibge]
df_sefaz_clean = raw_sefaz[colunas_interesse_sefaz]

# 3. Mesclar as tabelas baseando-se no Código do IBGE e no Ano correspondente
df_analise = pd.merge(df_ibge_clean, df_sefaz_clean, on=['cod_ibge', 'ano'], how='inner')

# Exibir informações da base consolidada
print(f"Tamanho final da base unificada: {df_analise.shape[0]} linhas e {df_analise.shape[1]} colunas.\n")
display(df_analise.head())

Tamanho final da base unificada: 2672 linhas e 12 colunas.



,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,tipo,valor
0,2400109,Acari,Central Potiguar,Caicó,2020,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,ICMS,3.116225e+06
1,2400109,Acari,Central Potiguar,Caicó,2020,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,IPVA,3.850488e+05
2,2400109,Acari,Central Potiguar,Caicó,2020,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,IPI,3.279713e+03
3,2400109,Acari,Central Potiguar,Caicó,2020,163709.383853,414.592918,15448.653756,53.373596,10–20 mil,ROYALTIES,4.459055e+04
4,2400109,Acari,Central Potiguar,Caicó,2021,162296.794114,440.079477,15315.352846,55.213740,10–20 mil,ICMS,3.576256e+06


### Análise de Qualidade dos Dados (`df_analise`)

Vamos executar um diagnóstico completo para garantir a integridade dos nossos dados antes de iniciar análises mais profundas.

In [6]:
print("=== 1. VERIFICAÇÃO DE VALORES NULOS ===")
missing_data = df_analise.isnull().sum()
display(missing_data[missing_data > 0] if (missing_data > 0).any() else "Nenhum valor nulo encontrado!")

print("\n=== 2. MUNICÍPIOS ÚNICOS POR ANO E TIPO DE REPASSE ===")
# Cada combinação de Ano e Tipo de repasse deve conter exatamente 167 municípios únicos
verificacao_temporal = df_analise.pivot_table(index='ano', columns='tipo', values='cod_ibge', aggfunc='nunique')
display(verificacao_temporal)

print("\n=== 3. RESUMO ESTATÍSTICO DOS VALORES FINANCEIROS ===")
display(df_analise[['pib_real_mil', 'valor']].describe().apply(lambda s: s.apply('{0:.2f}'.format)))

print("\n=== 4. ESTRUTURA E TIPOS DE COLUNA ===")
display(df_analise.dtypes)

=== 1. VERIFICAÇÃO DE VALORES NULOS ===


'Nenhum valor nulo encontrado!'


=== 2. MUNICÍPIOS ÚNICOS POR ANO E TIPO DE REPASSE ===


tipo,ICMS,IPI,IPVA,ROYALTIES
ano,,,,
2020,167,167,167,167
2021,167,167,167,167
2022,167,167,167,167
2023,167,167,167,167



=== 3. RESUMO ESTATÍSTICO DOS VALORES FINANCEIROS ===


,pib_real_mil,valor
count,2672.00,2672.00
mean,569042.16,2996218.98
std,2423132.38,17327459.90
min,26312.54,1481.30
25%,68229.48,26469.15
50%,135549.92,119649.93
75%,325580.50,1819671.64
max,31162043.00,373165581.80



=== 4. ESTRUTURA E TIPOS DE COLUNA ===


,0
cod_ibge,int64
municipio,object
mesorregiao,object
regiao_intermediaria,object
ano,int64
pib_real_mil,float64
rep_per_capita,float64
pib_per_capita,float64
pct_baixa_renda,float64
porte,object


### Formatando a Visualização de Valores Monetários

Vamos usar o styler do pandas para aplicar a formatação de moeda brasileira (`R$`) nas colunas financeiras.

In [7]:
# Criando um mapeamento de formatação para as colunas financeiras
# pib_real_mil está em milhares, então multiplicamos por 1000 se quisermos o valor real total ou formatamos mantendo a escala
formatacao = {
    'pib_real_mil': 'R$ {:,.2f} mil',
    'rep_per_capita': 'R$ {:,.2f}',
    'pib_per_capita': 'R$ {:,.2f}',
    'valor': 'R$ {:,.2f}'
}

# Exibindo as primeiras linhas do DataFrame com a formatação aplicada
df_analise.head().style.format(formatacao, thousands='.', decimal=',')

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,tipo,valor
0,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,373596",10–20 mil,ICMS,"R$ 3.116.225,09"
1,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,373596",10–20 mil,IPVA,"R$ 385.048,78"
2,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,373596",10–20 mil,IPI,"R$ 3.279,71"
3,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,373596",10–20 mil,ROYALTIES,"R$ 44.590,55"
4,2.400.109,Acari,Central Potiguar,Caicó,2.021,"R$ 162.296,79 mil","R$ 440,08","R$ 15.315,35","55,213740",10–20 mil,ICMS,"R$ 3.576.256,10"


In [8]:
# Atualizando o mapeamento de formatação para incluir pct_baixa_renda com o símbolo de porcentagem
formatacao_completa = {
    'pib_real_mil': 'R$ {:,.2f} mil',
    'rep_per_capita': 'R$ {:,.2f}',
    'pib_per_capita': 'R$ {:,.2f}',
    'valor': 'R$ {:,.2f}',
    'pct_baixa_renda': '{:.2f}%'
}

# Exibindo as primeiras linhas do DataFrame com todas as formatações brasileiras aplicadas
df_analise.head().style.format(formatacao_completa, thousands='.', decimal=',')

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,tipo,valor
0,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ICMS,"R$ 3.116.225,09"
1,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPVA,"R$ 385.048,78"
2,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPI,"R$ 3.279,71"
3,2.400.109,Acari,Central Potiguar,Caicó,2.020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ROYALTIES,"R$ 44.590,55"
4,2.400.109,Acari,Central Potiguar,Caicó,2.021,"R$ 162.296,79 mil","R$ 440,08","R$ 15.315,35","55,21%",10–20 mil,ICMS,"R$ 3.576.256,10"


In [9]:
# 1. Converter permanentemente as colunas 'cod_ibge' e 'ano' para string (texto) para evitar formatação numérica
df_analise['cod_ibge'] = df_analise['cod_ibge'].astype(str)
df_analise['ano'] = df_analise['ano'].astype(str)

# 2. Definir o mapeamento de formatação apenas para as colunas numéricas de interesse
formatacao_final = {
    'pib_real_mil': 'R$ {:,.2f} mil',
    'rep_per_capita': 'R$ {:,.2f}',
    'pib_per_capita': 'R$ {:,.2f}',
    'valor': 'R$ {:,.2f}',
    'pct_baixa_renda': '{:.2f}%'
}

# 3. Exibir o DataFrame estilizado
# Como as colunas 'cod_ibge' e 'ano' agora são strings, elas não serão afetadas por regras de milhares/decimais
df_analise.head().style.format(formatacao_final, thousands='.', decimal=',')

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,pib_real_mil,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,tipo,valor
0,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ICMS,"R$ 3.116.225,09"
1,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPVA,"R$ 385.048,78"
2,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPI,"R$ 3.279,71"
3,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163.709,38 mil","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ROYALTIES,"R$ 44.590,55"
4,2400109,Acari,Central Potiguar,Caicó,2021,"R$ 162.296,79 mil","R$ 440,08","R$ 15.315,35","55,21%",10–20 mil,ICMS,"R$ 3.576.256,10"


In [14]:
print(df_analise.info())
print(df_analise['pib_real_mil'].value_counts())
print(df_analise['ano'].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2672 entries, 0 to 2671
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   cod_ibge              2672 non-null   object 
 1   municipio             2672 non-null   object 
 2   mesorregiao           2672 non-null   object 
 3   regiao_intermediaria  2672 non-null   object 
 4   ano                   2672 non-null   object 
 5   pib_real_mil          2672 non-null   float64
 6   rep_per_capita        2672 non-null   float64
 7   pib_per_capita        2672 non-null   float64
 8   pct_baixa_renda       2672 non-null   float64
 9   porte                 2672 non-null   object 
 10  tipo                  2672 non-null   object 
 11  valor                 2672 non-null   float64
dtypes: float64(5), object(7)
memory usage: 250.6+ KB
None
pib_real_mil
49863.000000     4
163709.383853    4
162296.794114    4
181679.018623    4
195828.000000    4
            

### Criando Coluna do PIB em Milhões para Melhor Visualização

Vamos transformar os valores de milhares para milhões dividindo a coluna por 1.000, criando a variável `pib_real_milhoes`. Isso tornará a leitura dos dados muito mais intuitiva.

In [15]:
# 1. Criar a nova coluna expressando o PIB diretamente em milhões de reais
df_analise['pib_real_milhoes'] = df_analise['pib_real_mil'] / 1000.0

# 2. Atualizar o dicionário de formatação
formatacao_melhorada = {
    'pib_real_milhoes': 'R$ {:,.2f} milhões',
    'rep_per_capita': 'R$ {:,.2f}',
    'pib_per_capita': 'R$ {:,.2f}',
    'valor': 'R$ {:,.2f}',
    'pct_baixa_renda': '{:.2f}%'
}

# 3. Exibir as primeiras linhas com a nova formatação aplicada (ocultando a coluna antiga de milhares para simplificar)
colunas_exibicao = [
    'cod_ibge', 'municipio', 'mesorregiao', 'regiao_intermediaria',
    'ano', 'pib_real_milhoes', 'rep_per_capita', 'pib_per_capita',
    'pct_baixa_renda', 'porte', 'tipo', 'valor'
]

df_analise[colunas_exibicao].head().style.format(formatacao_melhorada, thousands='.', decimal=',')

,cod_ibge,municipio,mesorregiao,regiao_intermediaria,ano,pib_real_milhoes,rep_per_capita,pib_per_capita,pct_baixa_renda,porte,tipo,valor
0,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163,71 milhões","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ICMS,"R$ 3.116.225,09"
1,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163,71 milhões","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPVA,"R$ 385.048,78"
2,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163,71 milhões","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,IPI,"R$ 3.279,71"
3,2400109,Acari,Central Potiguar,Caicó,2020,"R$ 163,71 milhões","R$ 414,59","R$ 15.448,65","53,37%",10–20 mil,ROYALTIES,"R$ 44.590,55"
4,2400109,Acari,Central Potiguar,Caicó,2021,"R$ 162,30 milhões","R$ 440,08","R$ 15.315,35","55,21%",10–20 mil,ICMS,"R$ 3.576.256,10"


In [17]:
# Agrupando os valores de pib_real_milhoes por ano
pib_real_milhoes = df_analise.groupby('ano')['pib_real_milhoes']
print(pib_real_milhoes.describe())

      count        mean          std        min        25%         50%  \
ano                                                                      
2020  668.0  530.565268  2336.299567  26.312537  65.169825  123.567168   
2021  668.0  548.781694  2324.775733  26.561024  65.619605  129.828748   
2022  668.0  587.598469  2428.349967  26.749804  68.328883  143.046340   
2023  668.0  609.223192  2597.897972  29.717000  72.803000  155.095000   

             75%           max  
ano                             
2020  304.202188  28069.186929  
2021  310.268375  27644.746336  
2022  340.219849  28760.533354  
2023  348.751000  31162.043000  


In [20]:
# Encontrando outliers de forma global (todos os anos consolidados)
import numpy as np

# Usamos a série direta do DataFrame em vez do objeto groupby
serie_pib = df_analise['pib_real_milhoes']

# Calcula os quantis de forma correta
q1 = np.quantile(serie_pib, 0.25)
q3 = np.quantile(serie_pib, 0.75)
iqr = q3 - q1

# Calcula os limites inferior e superior
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# Filtra os registros que são considerados outliers
outliers = df_analise[(serie_pib < lower) | (serie_pib > upper)]

print(f"Limite inferior: R$ {lower:.2f}M")
print(f"Limite superior: R$ {upper:.2f}M")
print(f"Número de registros classificados como outliers: {len(outliers)}")

# Exibe os municípios que mais se destacam como outliers
display(outliers[['ano', 'municipio', 'pib_real_milhoes']].drop_duplicates().sort_values(by='pib_real_milhoes', ascending=False).head(10))

Limite inferior: R$ -317.80M
Limite superior: R$ 711.61M
Número de registros classificados como outliers: 340


,ano,municipio,pib_real_milhoes
1420,2023,Natal,31162.043000
1416,2022,Natal,28760.533354
1408,2020,Natal,28069.186929
1412,2021,Natal,27644.746336
1404,2023,Mossoró,10323.666000
1400,2022,Mossoró,10023.080843
1396,2021,Mossoró,9225.971299
1392,2020,Mossoró,8914.229656
568,2022,Parnamirim,7834.777459
564,2021,Parnamirim,7824.241210


In [21]:
# Agrupando os outliers para ver os municípios únicos (sem a repetição dos 4 tipos de repasse)
outliers_unicos = outliers[['ano', 'municipio', 'pib_real_milhoes']].drop_duplicates()

print(f"Total de registros de outliers na base longa: {len(outliers)}")
print(f"Total de combinações de Município-Ano que são outliers: {len(outliers_unicos)}")
print(f"Média de municípios outliers por ano: {len(outliers_unicos) / 4:.1f} municípios")

print("\nQuantidade de municípios outliers únicos por ano:")
print(outliers_unicos['ano'].value_counts().sort_index())

print("\nExemplo dos municípios considerados outliers em 2023 (limitado a 20 resultados para leitura):")
display(outliers_unicos[outliers_unicos['ano'] == '2023'].sort_values(by='pib_real_milhoes', ascending=False).head(20))

Total de registros de outliers na base longa: 340
Total de combinações de Município-Ano que são outliers: 85
Média de municípios outliers por ano: 21.2 municípios

Quantidade de municípios outliers únicos por ano:
ano
2020    20
2021    20
2022    23
2023    22
Name: count, dtype: int64

Exemplo dos municípios considerados outliers em 2023 (limitado a 20 resultados para leitura):


,ano,municipio,pib_real_milhoes
1420,2023,Natal,31162.043
1404,2023,Mossoró,10323.666
572,2023,Parnamirim,7608.526
2092,2023,São Gonçalo do Amarante,2588.564
796,2023,Guamaré,2581.922
1244,2023,Macaíba,2455.739
28,2023,Assú,2170.177
364,2023,Caicó,1735.797
1020,2023,João Câmara,1527.022
2124,2023,São José de Mipibu,1457.800
